Import Python libraries

In [8]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import sklearn as sk

print("pandas: ", pd.__version__)
print("numpy: ", np.__version__)
print("matplotlib: ", mpl.__version__)
print("sklearn: ", sk.__version__)

pandas:  3.0.6
numpy:  2.5.3
matplotlib:  3.11.2
sklearn:  1.9.1


Load dataset from archive


In [9]:
ds = pd.read_csv("datasets/archive/games.csv")
ds.shape
ds.info()
print("Null values: " % (ds.isnull().sum()))

<class 'pandas.DataFrame'>
RangeIndex: 51490 entries, 0 to 51489
Data columns (total 61 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   gameId              51490 non-null  int64
 1   creationTime        51490 non-null  int64
 2   gameDuration        51490 non-null  int64
 3   seasonId            51490 non-null  int64
 4   winner              51490 non-null  int64
 5   firstBlood          51490 non-null  int64
 6   firstTower          51490 non-null  int64
 7   firstInhibitor      51490 non-null  int64
 8   firstBaron          51490 non-null  int64
 9   firstDragon         51490 non-null  int64
 10  firstRiftHerald     51490 non-null  int64
 11  t1_champ1id         51490 non-null  int64
 12  t1_champ1_sum1      51490 non-null  int64
 13  t1_champ1_sum2      51490 non-null  int64
 14  t1_champ2id         51490 non-null  int64
 15  t1_champ2_sum1      51490 non-null  int64
 16  t1_champ2_sum2      51490 non-null  int64
 17  t1_c

In [10]:
fake_news = pd.read_csv("datasets/FakeNewsNet.csv")
#what are the columns and types?
fake_news.info()

#what does the data look like?
fake_news.head(10)
fake_news.sample(10, random_state=42)

#how much data  is missing?
print(f"Missing data: {fake_news.isna().mean().sort_values(ascending=False)}")

print(f"Missing values per column: {fake_news.isna().sum().sum()}")

for col in fake_news.columns:
    if fake_news[col].nunique() < 20:
        print(f"\n{col}:")
        print(fake_news[col].value_counts().head(10))

<class 'pandas.DataFrame'>
RangeIndex: 23196 entries, 0 to 23195
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   title          23196 non-null  str  
 1   news_url       22866 non-null  str  
 2   source_domain  22866 non-null  str  
 3   tweet_num      23196 non-null  int64
 4   real           23196 non-null  int64
dtypes: int64(2), str(3)
memory usage: 906.2 KB
Missing data: news_url         0.014227
source_domain    0.014227
title            0.000000
tweet_num        0.000000
real             0.000000
dtype: float64
Missing values per column: 660

real:
real
1    17441
0     5755
Name: count, dtype: int64


In [11]:
# 1. Is it unique?
print("Unique values:", fake_news["tweet_num"].nunique())
print("Total rows:", len(fake_news))

# 2. What's the range?
print(fake_news["tweet_num"].describe())

# 3. Is it sequential?
print(fake_news["tweet_num"].head(20))

# 4. Does it correlate with row order?
print(fake_news["tweet_num"].is_monotonic_increasing)

Unique values: 825
Total rows: 23196
count    23196.000000
mean        88.956803
std        488.694592
min          0.000000
25%         11.000000
50%         37.000000
75%         65.000000
max      29060.000000
Name: tweet_num, dtype: float64
0      42
1       0
2      63
3      20
4      38
5      45
6     124
7       4
8      59
9      10
10     18
11     95
12      4
13      0
14      0
15     63
16     54
17     64
18      1
19     19
Name: tweet_num, dtype: int64
False


New dataset: news_sample.csv

In [9]:
import glob
from collections import Counter
import pandas as pd

news = pd.read_csv("datasets/news_cleaned_2018_02_13.csv", nrows=0)
print(news.columns.tolist())

import json

f = "datasets/news_cleaned_2018_02_13.csv"
total_counts = Counter()

for chunk in pd.read_csv(
    f,
    chunksize=100_000,
    usecols=["type"],
    engine="python",
    on_bad_lines="skip",
    nrows=500_000
):
    total_counts.update(chunk["type"].value_counts().to_dict())
# Save to file
with open("datasets/full_type_distribution.json", "w") as out:
    json.dump(dict(total_counts), out, indent=2)

# Print
print("\nDataset type distribution:")
for label, count in total_counts.most_common():
    print(f"{label:15s} {count:>10,}")

#Check: how many distinct types does each domain have?
domains_per_type = news.groupby("domain")["type"].nunique()

# If every domain has exactly 1 type, this will all be 1s
print("Max types per domain:", domains_per_type.max())
print("Min types per domain:", domains_per_type.min())
print("Number of domains with more than 1 type:", (domains_per_type > 1).sum())

# Alternative view: for each domain, what types appear?
print("\nSample of domain → type mappings:")
print(news.groupby("domain")["type"].unique().head(20))


['Unnamed: 0', 'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 'meta_description', 'tags', 'summary', 'source']

Dataset type distribution:
political          129,562
conspiracy         101,758
bias                87,052
fake                70,030
rumor               21,660
junksci             14,418
clickbait           13,365
unknown             10,605
unreliable           6,210
satire               5,110
hate                 3,145
reliable             2,506
Max types per domain: nan
Min types per domain: nan
Number of domains with more than 1 type: 0

Sample of domain → type mappings:
Series([], Name: type, dtype: object)


In [3]:
import csv
import sys
import pandas as pd

csv.field_size_limit(sys.maxsize)

f = "datasets/news_cleaned_2018_02_13.csv"

chunks = []
for chunk in pd.read_csv(
    f,
    chunksize=100_000,
    usecols=["domain", "type"],
    engine="python",
    on_bad_lines="skip",
    nrows=500_000
):
    chunks.append(chunk)
   

sample = pd.concat(chunks, ignore_index=True)
print("Sample shape:", sample.shape)

domains_per_type = sample.groupby("domain")["type"].nunique()

print("\nMax types per domain:", domains_per_type.max())
print("Min types per domain:", domains_per_type.min())
print("Domains with more than 1 type:", (domains_per_type > 1).sum())
print("Total unique domains:", sample["domain"].nunique())

print("\nSample domain → type mappings:")
print(sample.groupby("domain")["type"].unique().head(20))

Sample shape: (500000, 2)

Max types per domain: 1
Min types per domain: 0
Domains with more than 1 type: 0
Total unique domains: 374

Sample domain → type mappings:
domain
100percentfedup.com                 [bias]
21stcenturywire.com           [conspiracy]
82.221.129.208                [conspiracy]
WideAwakeAmerica.com                 [nan]
aanirfan.blogspot.co.uk       [unreliable]
abovetopsecret.com            [conspiracy]
aceflashman.wordpress.com         [satire]
activistpost.com              [conspiracy]
actualidadpanamericana.com        [satire]
addictinginfo.org              [clickbait]
advocate.com                   [political]
aheadoftheherd.com                  [bias]
ahtribune.com                 [unreliable]
allnewspipeline.com           [unreliable]
alt-market.com                       [nan]
alternet.org                   [political]
americablog.com                     [bias]
americanborderpatrol.com            [hate]
americanfreepress.net         [conspiracy]
americanlo